In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


## PyTorch Implementation

1. Load and explore the data.
2. Split the data into training, validation, and test sets.
3. Build a model with fine-tuning of VGG16.
4. Train and evaluate the model.
5. Save the final model.


# Load Libraries and Set Up Data Paths


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.utils import make_grid
from tqdm.notebook import tqdm
import random
import warnings
from PIL import Image

warnings.filterwarnings('ignore')

In [ ]:
data_path = '../input/new-plant-diseases-dataset/'
train_folder = os.path.join(data_path, 'New Plant Diseases Dataset(Augmented)', 'New Plant Diseases Dataset(Augmented)', 'train')
valid_folder = os.path.join(data_path, 'New Plant Diseases Dataset(Augmented)', 'New Plant Diseases Dataset(Augmented)', 'valid')

classes = os.listdir(train_folder)
unique_plants = []
for item in classes:
    plant = item.split('_')[0]
    if plant not in unique_plants:
        unique_plants.append(plant)
print("Number of unique plants:", len(unique_plants))
print("Plants:", unique_plants)

In [ ]:
transform = transforms.Compose([transforms.Resize(128),transforms.ToTensor()])

dataset = ImageFolder(train_folder, transform=transform)
test_ds = ImageFolder(valid_folder, transform=transform)

print("Number of training images:", len(dataset))
print("Number of test images:", len(test_ds))
print("Number of classes:", len(dataset.classes))

In [ ]:
print("Classes:", dataset.classes)

In [ ]:
img, label = dataset[0]
print("Image shape:", img.shape)
print("Label:", label, dataset.classes[label])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.imshow(img.permute(1, 2, 0))
ax1.set_title("Original Image")
ax2.imshow(1 - img.permute(1, 2, 0))
ax2.set_title("Inverted Image")
plt.show()

### 1: Data Overview

Display 5 random images from the training dataset along with their labels.

In [ ]:
train_data_overview = DataLoader(dataset, batch_size=32, shuffle=True)
images, labels = next(iter(train_data_overview))
sample_indices = random.sample(range(len(images)), 5)

plt.figure(figsize=(20, 3))
for i, idx in enumerate(sample_indices):
    plt.subplot(1, 5, i+1)
    plt.imshow(images[idx].permute(1, 2, 0))
    plt.title(f"{dataset.classes[labels[idx]]}")
    plt.axis("off")
plt.suptitle("Slide 1: Data Overview\n\n\n\n\n\n\n\n\n\n\n\n\n5 Random Training Images.", fontsize=14)
plt.show()


## Data Splitting and Preparation for Training

We will split the data into a training (70%) and validation (30%) set, then prepare DataLoaders.


In [ ]:
random_seed = 42
torch.manual_seed(random_seed)
val_size = int(len(dataset) * 0.3)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

batch_size = 32  
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

for imgs, lbls in train_loader:
    plt.figure(figsize=(20, 8))
    plt.imshow(make_grid(imgs, nrow=16).permute(1, 2, 0))
    plt.axis('off')
    break

### 2: Data Distribution

Plot a bar chart showing the number of images per class to check if the data is balanced.


In [ ]:
diseases = os.listdir(train_folder)
class_counts = {cls: len(os.listdir(os.path.join(train_folder, cls))) for cls in diseases}
df = pd.DataFrame(list(class_counts.items()), columns=["Disease Class", "Number of Images"])
plt.figure(figsize=(15,5))
sns.barplot(data=df, x='Disease Class', y='Number of Images')
plt.xticks(rotation=90)
plt.title("Slide 2: Data Distribution\nNumber of Images per Disease Class.")
plt.show()

### Building the Model: Fine-Tuning VGG16 (PyTorch)

In this stage, we load the pre-trained VGG16 model, freeze most layers, and fine-tune the last 5 layers to adjust for 38 classes.


accuracy function

In [ ]:
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

In [ ]:
class ImageClassificationBase(nn.Module):
    def training_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        return loss

    def validation_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        acc = accuracy(out, labels)
        return {'val_loss': loss, 'val_acc': acc}

    def validation_epoch_end(self, outputs):
        batch_loss = [x['val_loss'] for x in outputs]
        epoch_loss = torch.stack(batch_loss).mean()
        batch_acc = [x['val_acc'] for x in outputs]
        epoch_acc = torch.stack(batch_acc).mean()
        return {'val_loss': epoch_loss.item(), 'val_acc': epoch_acc.item()}

    def epoch_end(self, epoch, result):
        print(f"Epoch [{epoch}], train_loss: {result['train_loss']:.4f}, val_loss: {result['val_loss']:.4f}, val_acc: {result['val_acc']:.4f}")

### VGG16 model

In [ ]:
class Plant_Disease_VGG16(ImageClassificationBase):
    def __init__(self):
        super().__init__()
        self.network = models.vgg16(pretrained=True)
        for param in list(self.network.features.parameters())[:-5]:
            param.requires_grad = False
        num_ftrs = self.network.classifier[-1].in_features
        self.network.classifier[-1] = nn.Linear(num_ftrs, 38)
        
    def forward(self, xb):
        return self.network(xb)

### 3: Model Architecture

Display the fine-tuned VGG16 model structure.  
The model shows that most layers are frozen, with only the last 5 layers and the classifier being trainable.

In [ ]:
def get_default_device():
    return torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
def to_device(data, device):
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)
class DeviceDataLoader():
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device
    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)
    def __len__(self):
        return len(self.dl)

device = get_default_device()
print("Using device:", device)

In [ ]:
train_loader = DeviceDataLoader(train_loader, device)
val_loader = DeviceDataLoader(val_loader, device)
test_loader = DeviceDataLoader(test_loader, device)

vgg_model_ft = to_device(Plant_Disease_VGG16(), device)
print(vgg_model_ft)

### 4: Training and Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader):
    model.eval()
    outputs = [model.validation_step(batch) for batch in val_loader]
    return model.validation_epoch_end(outputs)

def fit(epochs, lr, model, train_loader, val_loader, opt_func=torch.optim.SGD):
    history = []
    optimizer = opt_func(model.parameters(), lr)
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            loss = model.training_step(batch)
            train_losses.append(loss)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        torch.cuda.empty_cache()
        result = evaluate(model, val_loader)
        result['train_loss'] = torch.stack(train_losses).mean().item()
        model.epoch_end(epoch, result)
        history.append(result)
    return history

history_vgg_ft = fit(10, 0.0001, vgg_model_ft, train_loader, val_loader, opt_func=torch.optim.Adam)

In [ ]:
def plot_losses(history):
    train_losses = [x.get('train_loss') for x in history]
    val_losses = [x['val_loss'] for x in history]
    plt.plot(train_losses, '-bx', label="Train Loss")
    plt.plot(val_losses, '-rx', label="Val Loss")
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss vs. Epochs')
    plt.show()

def plot_accuracies(history):
    accuracies = [x['val_acc'] for x in history]
    plt.plot(accuracies, '-x')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Accuracy vs. Epochs')
    plt.show()

plot_accuracies(history_vgg_ft)
plot_losses(history_vgg_ft)

print("Evaluation on Test Set (Fine-tuned VGG16):")
print(evaluate(vgg_model_ft, test_loader))

### 5: Model Predictions

Display 5 sample test images with their true and predicted labels.


In [ ]:
test_sample_indices = random.sample(range(len(test_ds.imgs)), 5)
plt.figure(figsize=(20, 5))

for i, idx in enumerate(test_sample_indices):
    img_path, _ = test_ds.imgs[idx]
    img = Image.open(img_path)
    img = transform(img)

    img = img.unsqueeze(0).to(device)

    prediction = vgg_model_ft(img)
    pred_class = dataset.classes[torch.argmax(prediction)]
    true_label = img_path.split('/')[-2]

    img_cpu = img.squeeze(0).cpu()

    plt.subplot(1, 5, i + 1)
    plt.imshow(img_cpu.permute(1, 2, 0).numpy())
    plt.title(f"True: {true_label}\nPred: {pred_class}")
    plt.axis("off")

plt.suptitle("Slide 5: Model Predictions\n5 Test Images with True and Predicted Labels.", fontsize=14)
plt.show()

In [ ]:
def calculate_accuracy(test_loader, model):
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total * 100
    return accuracy

accuracy = calculate_accuracy(test_loader, vgg_model_ft)
print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
torch.save(vgg_model_ft.state_dict(), '/kaggle/working/vgg_ft_model.pth')

# Description

This notebook provides a complete pipeline for training a plant disease classification model using PyTorch. The steps covered in this notebook are:

1. **Data Exploration and Preprocessing**:
   - Loading the plant disease dataset.
   - Visualizing random images from the training set.
   - Checking data distribution across different disease classes.

2. **Data Splitting and Preparation**:
   - Splitting the dataset into training, validation, and test sets.
   - Preparing DataLoader objects for efficient batching and shuffling.

3. **Model Building**:
   - Using a pre-trained VGG16 model and fine-tuning the last layers to classify 38 different plant diseases.
   - Freezing most of the layers to retain pre-trained features and adjusting the classifier for the target number of classes.

4. **Model Training**:
   - Training the model on the training dataset.
   - Evaluating the model's performance on the validation set during each epoch.

5. **Model Evaluation**:
   - Visualizing predictions made by the model on sample test images.
   - Calculating and displaying the final test accuracy.

6. **Model Saving**:
   - Saving the trained model to disk for future use.